# Healthcare Claims EDA & Cost Analysis
**CMS DE-SynPUF Medicare Claims | 2008-2010 | 230M+ Records**

---

**Objective:** Answer six core business questions a US health plan asks every quarter - where the money goes, for whom, and what's avoidable - using 230 million synthetic Medicare claims processed with DuckDB.

| # | Business Question |
|---|---|
| 1 | What share of total spend comes from the costliest members? |
| 2 | What is the PMPM cost by service line, and how does it trend? |
| 3 | How does chronic-condition layering drive cost? |
| 4 | What is the 30-day readmission rate, and what drives it? |
| 5 | How does cost vary by age and sex? |
| 6 | What are the top cost drivers by DRG, HCPCS, and drug category? |

## 0 - Setup

In [103]:
import sys, warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# src/ on path
sys.path.insert(0, str(Path.cwd().parent))

from src.config          import *
from src.data_loader     import get_connection, create_views, create_beneficiary_enriched, row_count
from src.quality         import compute_completeness, print_completeness_tiers, check_referential_integrity, check_temporal_consistency
from src.visual_identity import (
    visual_identity, fig_title, adjust_axis, auto_scale_axis,
    fmt_num, fmt_usd, bar_labels, render_table,
    ECON_BG, ECON_RED, NATURE_TEAL, NATURE_CORAL, DEEP_BLUE,
    AMBER, PLUM, WARM_GREY, CHARCOAL, LIGHT_GREY,
    PALETA_SEQ, WHITE
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# DuckDB connection & views─
con = get_connection(memory_limit="6GB", threads=4)
create_views(con)

print("✓ Views created for:", list(DESYNPUF_PATHS.keys()))
print(f"✓ DuckDB ready  ·  memory_limit = 6 GB  ·  threads = 4")

✓ Views created for: ['beneficiary', 'inpatient', 'outpatient', 'carrier', 'pde']
✓ DuckDB ready  ·  memory_limit = 6 GB  ·  threads = 4


In [ ]:
# Artifact store and DuckDB resource settings
# The 6 GB limit set in 0 stays the notebook default; heavy cells raise it only
# inside `with duckdb_settings(con, **HEAVY):` and it is restored on exit.
import os
from src.artifacts import ArtifactStore, duckdb_settings

ARTIFACTS_DIR = Path.cwd().parent / "artifacts"
SPILL_DIR     = ARTIFACTS_DIR / "_duckdb_spill"
SPILL_DIR.mkdir(parents=True, exist_ok=True)

# 70% of physical RAM (Linux sysconf); never below the 6 GB already in use
TOTAL_RAM_GB = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
HEAVY_MEM_GB = max(6, int(TOTAL_RAM_GB * 0.70))

# temp_directory is set once for the whole session, before any spill happens,
# and is NOT part of HEAVY: DuckDB refuses to switch it after spill files were
# written there ("Cannot switch temporary directory after the current one has
# been used"), so a per-cell set/restore cannot work.
_spill_literal = str(SPILL_DIR).replace("'", "''")
try:
    con.execute(f"SET temp_directory = '{_spill_literal}'")
except Exception as exc:
    print(f"[duckdb] temp_directory not changed (already used in this session): {exc}")
_current_tmp = con.execute("SELECT current_setting('temp_directory')").fetchone()[0]
print(f"[duckdb] temp_directory = {_current_tmp}")

HEAVY = dict(
    memory_limit=f"{HEAVY_MEM_GB}GB",
    threads=2,
    preserve_insertion_order=False,
)
print(f"Physical RAM: {TOTAL_RAM_GB:,.1f} GB | heavy memory_limit: {HEAVY_MEM_GB} GB | spill: {SPILL_DIR}")

store = ArtifactStore(con, ARTIFACTS_DIR)


def _dt(col):
    # Same BIGINT YYYYMMDD -> DATE parsing used elsewhere in the notebook
    return (f"COALESCE(TRY_CAST({col} AS DATE), "
            f"TRY_STRPTIME(CAST({col} AS VARCHAR), '%Y%m%d')::DATE)")


def _show_df(title, df):
    # Single-row results print one metric per line; multi-row results as a table
    print(f"\n  [{title}]")
    if len(df) == 1:
        for k, v in df.iloc[0].items():
            if isinstance(v, (int, float, np.integer, np.floating)) and pd.notna(v):
                print(f"    {k:<42s} {v:>16,.0f}")
            else:
                print(f"    {k:<42s} {str(v):>16s}")
    else:
        print(df.to_string(index=False))

[duckdb] temp_directory = /home/aosousa/Documents/PORTFOLIO/artifacts/_duckdb_spill
Physical RAM: 19.4 GB | heavy memory_limit: 13 GB | spill: /home/aosousa/Documents/PORTFOLIO/artifacts/_duckdb_spill
[artifacts] store at /home/aosousa/Documents/PORTFOLIO/artifacts | 47 artifact(s) in manifest


## 1 - Data Quality Assessment

Before any analysis, we validate the data across three dimensions:
1. **Volume** - do row counts match expectations?
2. **Completeness** - what's the fill rate per column?
3. **Referential integrity** - do foreign keys resolve?
4. **Temporal consistency** - are date ranges plausible?

In [32]:
# 1a. Volume check
print("TABLE VOLUME CHECK")
print("=" * 60)
for table, expected in EXPECTED_ROWS.items():
    actual = row_count(con, table)
    status = "✓" if actual == expected else "✗"
    delta  = actual - expected
    print(f"  {status} {table:<14s}  {actual:>14,}  (expected {expected:>14,}, Δ {delta:+,})")

TABLE VOLUME CHECK
  ✓ beneficiary          6,873,274  (expected      6,873,274, Δ +0)
  ✓ carrier             94,863,452  (expected     94,863,452, Δ +0)
  ✓ inpatient            1,332,822  (expected      1,332,822, Δ +0)
  ✓ outpatient          15,826,987  (expected     15,826,987, Δ +0)
  ✓ pde                111,085,969  (expected    111,085,969, Δ +0)


In [33]:
# 1b. Completeness by table
for table in ["beneficiary", "inpatient", "outpatient", "carrier", "pde"]:
    print(f"\n{'─'*60}")
    print(f"COMPLETENESS: {table.upper()}")
    print(f"{'─'*60}")
    df_c = compute_completeness(con, table, exclude_prefixes=[])
    print_completeness_tiers(df_c, max_per_tier=20)


────────────────────────────────────────────────────────────
COMPLETENESS: BENEFICIARY
────────────────────────────────────────────────────────────
Completeness summary (34 columns):

  100% complete: 33 columns
    DESYNPUF_ID                              100.00%
    BENE_BIRTH_DT                            100.00%
    BENE_SEX_IDENT_CD                        100.00%
    BENE_RACE_CD                             100.00%
    BENE_ESRD_IND                            100.00%
    SP_STATE_CODE                            100.00%
    BENE_COUNTY_CD                           100.00%
    BENE_HI_CVRAGE_TOT_MONS                  100.00%
    BENE_SMI_CVRAGE_TOT_MONS                 100.00%
    BENE_HMO_CVRAGE_TOT_MONS                 100.00%
    PLAN_CVRG_MOS_NUM                        100.00%
    SP_ALZHDMTA                              100.00%
    SP_CHF                                   100.00%
    SP_CHRNKIDN                              100.00%
    SP_CNCR                                  

In [34]:
# 1c. Referential integrity
print("REFERENTIAL INTEGRITY CHECKS")
print("=" * 60)

ri_checks = [
    ("inpatient",  "beneficiary", "DESYNPUF_ID", "DESYNPUF_ID", "Inpatient → Beneficiary"),
    ("outpatient", "beneficiary", "DESYNPUF_ID", "DESYNPUF_ID", "Outpatient → Beneficiary"),
    ("carrier",    "beneficiary", "DESYNPUF_ID", "DESYNPUF_ID", "Carrier → Beneficiary"),
    ("pde",        "beneficiary", "DESYNPUF_ID", "DESYNPUF_ID", "PDE → Beneficiary"),
]

for left, right, lk, rk, desc in ri_checks:
    check_referential_integrity(con, left, right, lk, rk, desc)

REFERENTIAL INTEGRITY CHECKS


In [35]:
# 1d. Temporal consistency
print("TEMPORAL CONSISTENCY CHECKS")
print("=" * 60)

tc_checks = [
    ("inpatient",  "CLM_FROM_DT",  "CLM_THRU_DT",  "2008-01-01", "2010-12-31"),
    ("outpatient", "CLM_FROM_DT",  "CLM_THRU_DT",  "2008-01-01", "2010-12-31"),
    ("carrier",    "CLM_FROM_DT",  "CLM_THRU_DT",  "2008-01-01", "2010-12-31"),
    ("pde",        "SRVC_DT",       None,           "2008-01-01", "2010-12-31"),
]

for table, date_from, date_thru, exp_min, exp_max in tc_checks:
    if date_thru:
        check_temporal_consistency(con, table, date_from, date_thru, exp_min, exp_max)
    else:
        # PDE has only a single service date (SRVC_DT).
        # DE-SynPUF stores dates as YYYYMMDD integers (BIGINT), so we
        # cast to VARCHAR then parse to DATE before comparing to date literals.
        date_cast = (
            f"COALESCE(TRY_CAST({date_from} AS DATE), "
            f"TRY_STRPTIME(CAST({date_from} AS VARCHAR), '%Y%m%d')::DATE)"
        )
        result = con.execute(f"""
            SELECT
                MIN({date_cast})::VARCHAR AS min_dt,
                MAX({date_cast})::VARCHAR AS max_dt,
                SUM(CASE WHEN {date_cast} < DATE '{exp_min}'
                           OR {date_cast} > DATE '{exp_max}'
                         THEN 1 ELSE 0 END) AS out_of_range
            FROM {table}
            WHERE {date_from} IS NOT NULL
        """).fetchdf()
        r = result.iloc[0]
        oor = int(r["out_of_range"])
        status = "✓" if oor == 0 else f"✗ ({oor:,} out of range)"
        print(f"  {status}  {table}.{date_from}: [{r['min_dt']} ... {r['max_dt']}]")

TEMPORAL CONSISTENCY CHECKS


  ✓  pde.SRVC_DT: [2008-01-01 ... 2010-12-31]


> **Data quality verdict:** The DE-SynPUF dataset is synthetic by design - CMS generated it for research and education.<br>
We expect full referential integrity across all joins and dates within the 2008-2010 window.<br>
Completeness gaps in optional fields (e.g., diagnosis codes 4-10, secondary procedures) reflect clinical reality.<br>
Not every claim carries the maximum number of codes.

In [105]:
# Typed claim bases for INPATIENT and OUTPATIENT
# Dates are parsed once, CLM_ID/PRVDR_NUM are cast to VARCHAR, and every row gets
# claim_key = CLM_ID || '-' || SEGMENT.
# CLM_ID can carry two rows with different PRVDR_NUM, so CLM_ID alone is not a claim identifier.
# (CLM_ID, SEGMENT) is the atomic record. Uniqueness is asserted bELOW.
# Only the columns needed for family netting, stays and DRG ranking are kept.

CLM_IP_SQL = f"""
    SELECT
        CAST(CLM_ID AS VARCHAR) || '-' || CAST(SEGMENT AS VARCHAR) AS claim_key,
        CAST(CLM_ID AS VARCHAR)            AS CLM_ID,
        CAST(SEGMENT AS INTEGER)           AS SEGMENT,
        DESYNPUF_ID,
        CAST(PRVDR_NUM AS VARCHAR)         AS PRVDR_NUM,
        {_dt('CLM_ADMSN_DT')}              AS adm_dt,
        {_dt('CLM_FROM_DT')}               AS from_dt,
        {_dt('CLM_THRU_DT')}               AS thru_dt,
        {_dt('NCH_BENE_DSCHRG_DT')}        AS dschrg_dt,
        CLM_PMT_AMT,
        NCH_PRMRY_PYR_CLM_PD_AMT,
        CAST(CLM_DRG_CD AS VARCHAR)        AS CLM_DRG_CD
    FROM inpatient
"""

CLM_OP_SQL = f"""
    SELECT
        CAST(CLM_ID AS VARCHAR) || '-' || CAST(SEGMENT AS VARCHAR) AS claim_key,
        CAST(CLM_ID AS VARCHAR)            AS CLM_ID,
        CAST(SEGMENT AS INTEGER)           AS SEGMENT,
        DESYNPUF_ID,
        CAST(PRVDR_NUM AS VARCHAR)         AS PRVDR_NUM,
        {_dt('CLM_FROM_DT')}               AS from_dt,
        {_dt('CLM_THRU_DT')}               AS thru_dt,
        CLM_PMT_AMT,
        NCH_PRMRY_PYR_CLM_PD_AMT
    FROM outpatient
"""

with duckdb_settings(con, **HEAVY):
    store.materialize("clm_ip", CLM_IP_SQL, inputs=f"{DESYNPUF_PATHS['inpatient']}/*.csv") # materialize as csv for further inspection
    store.materialize("clm_op", CLM_OP_SQL, inputs=f"{DESYNPUF_PATHS['outpatient']}/*.csv") # materialize as csv for further inspection

    # Validation assumptions
    # 1) row count must equal the source;
    # 2) claim_key must be unique; and
    # 3) payment total must equal the source total (no row lost or added)
    for name, src_tbl in [("clm_ip", "inpatient"), ("clm_op", "outpatient")]:
        v = con.execute(f"""
            SELECT COUNT(*)                                  AS n_rows,
                   COUNT(DISTINCT claim_key)                 AS n_claim_key,
                   COUNT(*) FILTER (WHERE from_dt IS NULL)   AS from_dt_null,
                   COUNT(*) FILTER (WHERE thru_dt IS NULL)   AS thru_dt_null,
                   SUM(CLM_PMT_AMT)                          AS sum_pmt
            FROM {name}
        """).fetchdf().iloc[0]
        src_sum = con.execute(f"SELECT SUM(CLM_PMT_AMT) FROM {src_tbl}").fetchone()[0]
        print(f"\n  VALIDATION {name}")
        print(f"    rows                {int(v['n_rows']):>16,}  (expected {EXPECTED_ROWS[src_tbl]:,})")
        print(f"    distinct claim_key  {int(v['n_claim_key']):>16,}")
        print(f"    from_dt NULL        {int(v['from_dt_null']):>16,}")
        print(f"    thru_dt NULL        {int(v['thru_dt_null']):>16,}")
        print(f"    sum CLM_PMT_AMT     {v['sum_pmt']:>16,.2f}  (source {src_sum:,.2f})")
        assert int(v["n_rows"]) == EXPECTED_ROWS[src_tbl], f"{name}: row count differs from source"
        assert int(v["n_rows"]) == int(v["n_claim_key"]), f"{name}: claim_key is not unique"
        assert abs(float(v["sum_pmt"]) - float(src_sum)) < 0.005, f"{name}: payment total differs from source"
    print("\n  ✓ typed bases validated")

[duckdb] SET memory_limit = '13GB'  (was '5.5 GiB')
[duckdb] SET threads = 2  (was 4)
[duckdb] SET preserve_insertion_order = False  (was True)
[artifacts] HIT   clm_ip                                   rows=     1,332,822  built=2026-11-18T07:52:18  key=dbb3dffbbb92
[artifacts] HIT   clm_op                                   rows=    15,826,987  built=2026-11-18T07:53:24  key=a77cb164598e

  VALIDATION clm_ip
    rows                       1,332,822  (expected 1,332,822)
    distinct claim_key         1,332,822
    from_dt NULL                   1,289
    thru_dt NULL                   1,289
    sum CLM_PMT_AMT     12,710,862,270.00  (source 12,710,862,270.00)

  VALIDATION clm_op
    rows                      15,826,987  (expected 15,826,987)
    distinct claim_key        15,826,987
    from_dt NULL                 222,803
    thru_dt NULL                 222,803
    sum CLM_PMT_AMT     4,482,288,370.00  (source 4,482,288,370.00)

  ✓ typed bases validated
[duckdb] restored memory_lim

In [106]:
#1f. Family-rule calibration diagnostics, rebuilt on the typed bases

_PAIRS_CTE = """
    multi AS (
        SELECT CLM_ID FROM {t} GROUP BY CLM_ID HAVING COUNT(*) > 1
    ),
    p AS (
        SELECT CLM_ID,
               COUNT(*)                                    AS n_rows,
               COUNT(DISTINCT DESYNPUF_ID)                 AS n_bene,
               COUNT(DISTINCT PRVDR_NUM)                   AS n_prvdr,
               COUNT(DISTINCT from_dt)                     AS n_from,
               COUNT(DISTINCT thru_dt)                     AS n_thru,
               MAX(CLM_PMT_AMT) FILTER (WHERE SEGMENT = 1) AS pmt1,
               MAX(CLM_PMT_AMT) FILTER (WHERE SEGMENT = 2) AS pmt2
        FROM {t}
        WHERE CLM_ID IN (SELECT CLM_ID FROM multi)
        GROUP BY CLM_ID
    )
"""

DIAG = {
    # SEGMENT values present
    "seg_dist": ("""
        SELECT SEGMENT, COUNT(*) AS n_rows FROM {t} GROUP BY SEGMENT
    """, "SEGMENT"),

    # CLM_IDs with no SEGMENT=1 row
    "orphan_segments": ("""
        SELECT COUNT(*)              AS clm_ids_without_segment_1,
               COALESCE(SUM(pmt), 0) AS pmt_on_orphan_segments
        FROM (SELECT CLM_ID, SUM(CLM_PMT_AMT) AS pmt
              FROM {t} GROUP BY CLM_ID HAVING MIN(SEGMENT) > 1)
    """, None),

    # Segment-pair profile (same metrics as the inpatient run, plus row-count check)
    "segment_pairs": ("WITH" + _PAIRS_CTE + """
        SELECT COUNT(*)                                                  AS pairs,
               COUNT(*) FILTER (WHERE n_rows <> 2)                       AS clm_ids_not_exactly_2_rows,
               COUNT(*) FILTER (WHERE n_bene = 1)                        AS same_bene,
               COUNT(*) FILTER (WHERE n_prvdr = 1)                       AS same_prvdr,
               COUNT(*) FILTER (WHERE n_from = 1 AND n_thru = 1)         AS same_from_and_thru,
               COUNT(*) FILTER (WHERE pmt1 IS NULL OR pmt2 IS NULL)      AS not_a_1_2_pair,
               COUNT(*) FILTER (WHERE pmt1 = 0 AND pmt2 = 0)             AS both_zero,
               COUNT(*) FILTER (WHERE pmt1 = pmt2 AND pmt1 <> 0)         AS identical_nonzero,
               COUNT(*) FILTER (WHERE pmt1 <> 0 AND pmt2 = -pmt1)        AS exact_mirror_reversal,
               COUNT(*) FILTER (WHERE pmt2 < 0 AND pmt2 <> -pmt1)        AS seg2_negative_partial,
               COUNT(*) FILTER (WHERE pmt1 < 0)                          AS seg1_negative,
               COUNT(*) FILTER (WHERE pmt2 = 0 AND pmt1 <> 0)            AS seg2_zero_only,
               COUNT(*) FILTER (WHERE pmt1 > 0 AND pmt2 > 0 AND pmt1 <> pmt2) AS both_positive_different
        FROM p
    """, None),

    # Cross-tab - provider agreement x payment relation, with dollars per cell.
    "segment_pairs_xtab": ("WITH" + _PAIRS_CTE + """
        SELECT (n_prvdr = 1)                                             AS same_prvdr,
               CASE
                   WHEN pmt1 IS NULL OR pmt2 IS NULL THEN 'not_a_1_2_pair'
                   WHEN pmt1 = 0 AND pmt2 = 0        THEN 'both_zero'
                   WHEN pmt1 = pmt2                  THEN 'identical_nonzero'
                   WHEN pmt2 = -pmt1                 THEN 'exact_mirror'
                   WHEN pmt1 < 0 OR pmt2 < 0         THEN 'one_negative'
                   WHEN pmt1 = 0 OR pmt2 = 0         THEN 'one_zero'
                   ELSE 'both_positive_different'
               END                                                       AS pmt_relation,
               COUNT(*)                                                  AS pairs,
               SUM(pmt1)                                                 AS sum_pmt_seg1,
               SUM(pmt2)                                                 AS sum_pmt_seg2
        FROM p
        GROUP BY ALL
    """, "same_prvdr, pmt_relation"),

    # Counterparts of negative rows (mutually exclusive cascade)
    "negative_counterparts": ("""
        WITH neg AS (
            SELECT claim_key, CLM_ID, DESYNPUF_ID, PRVDR_NUM, from_dt, thru_dt, CLM_PMT_AMT
            FROM {t} WHERE CLM_PMT_AMT < 0
        ),
        pos AS (
            SELECT CLM_ID, DESYNPUF_ID, PRVDR_NUM, from_dt, thru_dt, CLM_PMT_AMT
            FROM {t} WHERE CLM_PMT_AMT > 0
        ),
        m AS (
            SELECT n.claim_key,
                   ANY_VALUE(n.CLM_PMT_AMT) AS neg_amt,
                   MAX(CASE WHEN p.CLM_ID = n.CLM_ID AND p.CLM_PMT_AMT = -n.CLM_PMT_AMT THEN 1 ELSE 0 END) AS f1,
                   MAX(CASE WHEN p.CLM_ID <> n.CLM_ID AND p.CLM_PMT_AMT = -n.CLM_PMT_AMT
                             AND p.from_dt = n.from_dt AND p.thru_dt = n.thru_dt THEN 1 ELSE 0 END)      AS f2,
                   MAX(CASE WHEN p.from_dt = n.from_dt AND p.thru_dt = n.thru_dt THEN 1 ELSE 0 END)      AS f3,
                   MAX(CASE WHEN p.CLM_ID IS NOT NULL THEN 1 ELSE 0 END)                                 AS f4
            FROM neg n
            LEFT JOIN pos p
              ON  p.DESYNPUF_ID = n.DESYNPUF_ID
              AND p.PRVDR_NUM   = n.PRVDR_NUM
            GROUP BY n.claim_key
        )
        SELECT COUNT(*)                                                         AS negative_rows,
               COALESCE(SUM(neg_amt), 0)                                        AS negative_dollars,
               COUNT(*) FILTER (WHERE f1 = 1)                                   AS c1_mirror_same_clm_id,
               COUNT(*) FILTER (WHERE f1 = 0 AND f2 = 1)                        AS c2_mirror_other_clm_id,
               COUNT(*) FILTER (WHERE f1 = 0 AND f2 = 0 AND f3 = 1)             AS c3_partial_same_dates,
               COUNT(*) FILTER (WHERE f1 = 0 AND f2 = 0 AND f3 = 0 AND f4 = 1)  AS c4_same_bene_prvdr_other_dates,
               COALESCE(SUM(neg_amt) FILTER (WHERE f1 = 0 AND f2 = 0 AND f3 = 0 AND f4 = 1), 0) AS c4_dollars,
               COUNT(*) FILTER (WHERE f4 = 0)                                   AS c5_no_counterpart,
               COALESCE(SUM(neg_amt) FILTER (WHERE f4 = 0), 0)                  AS c5_dollars
        FROM m
    """, None),

    # Exact duplicates under different CLM_IDs
    "exact_duplicates": ("""
        SELECT COUNT(*)                                                  AS dup_groups,
               COALESCE(SUM(n - 1), 0)                                   AS surplus_claims,
               COUNT(*) FILTER (WHERE CLM_PMT_AMT > 0)                   AS groups_positive_pmt,
               COUNT(*) FILTER (WHERE CLM_PMT_AMT = 0)                   AS groups_zero_pmt,
               COUNT(*) FILTER (WHERE CLM_PMT_AMT < 0)                   AS groups_negative_pmt,
               COALESCE(SUM(CLM_PMT_AMT * (n - 1)) FILTER (WHERE CLM_PMT_AMT > 0), 0) AS surplus_dollars_positive
        FROM (
            SELECT DESYNPUF_ID, PRVDR_NUM, from_dt, thru_dt, CLM_PMT_AMT,
                   COUNT(DISTINCT CLM_ID) AS n
            FROM {t}
            GROUP BY DESYNPUF_ID, PRVDR_NUM, from_dt, thru_dt, CLM_PMT_AMT
            HAVING COUNT(DISTINCT CLM_ID) > 1
        )
    """, None),

    # Zero-paid rows
    "zero_paid_split": ("""
        SELECT COUNT(*)                                                         AS zero_pmt_rows,
               COUNT(*) FILTER (WHERE NCH_PRMRY_PYR_CLM_PD_AMT > 0)             AS other_primary_payer_paid,
               COUNT(*) FILTER (WHERE COALESCE(NCH_PRMRY_PYR_CLM_PD_AMT, 0) <= 0) AS nobody_paid
        FROM {t}
        WHERE CLM_PMT_AMT = 0
    """, None),
}

# Inpatient overlap anatomy
OVERLAP_SQL = """
    WITH s AS (
        SELECT claim_key, CLM_ID, PRVDR_NUM, adm_dt, from_dt, thru_dt,
               LAG(CLM_ID)    OVER w AS prev_clm,
               LAG(PRVDR_NUM) OVER w AS prev_prvdr,
               LAG(adm_dt)    OVER w AS prev_adm,
               LAG(thru_dt)   OVER w AS prev_thru
        FROM clm_ip
        WINDOW w AS (PARTITION BY DESYNPUF_ID ORDER BY from_dt, CLM_ID, SEGMENT)
    )
    SELECT COUNT(*)                                                               AS overlapping,
           COUNT(*) FILTER (WHERE CLM_ID = prev_clm)                              AS same_clm_id_segment,
           COUNT(*) FILTER (WHERE CLM_ID <> prev_clm AND adm_dt = prev_adm)       AS same_admission_date,
           COUNT(*) FILTER (WHERE CLM_ID <> prev_clm AND adm_dt = prev_adm
                              AND PRVDR_NUM = prev_prvdr)                         AS same_admission_same_prvdr,
           COUNT(*) FILTER (WHERE CLM_ID <> prev_clm AND PRVDR_NUM = prev_prvdr)  AS same_provider,
           COUNT(*) FILTER (WHERE CLM_ID <> prev_clm AND thru_dt <= prev_thru)    AS nested_inside_previous
    FROM s
    WHERE from_dt < prev_thru
"""

print("FAMILY-RULE CALIBRATION DIAGNOSTICS (typed bases)")
print("=" * 70)
diag = {}
with duckdb_settings(con, **HEAVY):
    for t in ["clm_ip", "clm_op"]:
        for key, (sql_tpl, order_by) in DIAG.items():
            name = f"diag_{t}_{key}"
            diag[name] = store.query_df(name, sql_tpl.format(t=t), deps=[t], order_by=order_by)
            _show_df(name, diag[name])
    diag["diag_clm_ip_overlap_anatomy"] = store.query_df(
        "diag_clm_ip_overlap_anatomy", OVERLAP_SQL, deps=["clm_ip"])
    _show_df("diag_clm_ip_overlap_anatomy", diag["diag_clm_ip_overlap_anatomy"])

FAMILY-RULE CALIBRATION DIAGNOSTICS (typed bases)
[duckdb] SET memory_limit = '13GB'  (was '5.5 GiB')
[duckdb] SET threads = 2  (was 4)
[duckdb] SET preserve_insertion_order = False  (was True)
[artifacts] HIT   diag_clm_ip_seg_dist                     rows=             2  built=2026-11-18T07:54:12  key=869038809d14

  [diag_clm_ip_seg_dist]
 SEGMENT  n_rows
       1 1331533
       2    1289
[artifacts] HIT   diag_clm_ip_orphan_segments              rows=             1  built=2026-11-18T07:54:14  key=9c3446a57ae0

  [diag_clm_ip_orphan_segments]
    clm_ids_without_segment_1                                 0
    pmt_on_orphan_segments                                    0
[artifacts] HIT   diag_clm_ip_segment_pairs                rows=             1  built=2026-11-18T07:54:15  key=545ef61ea012

  [diag_clm_ip_segment_pairs]
    pairs                                                 1,289
    clm_ids_not_exactly_2_rows                                0
    same_bene                        

In [107]:
# 1g. Carrier and PDE repetition, sign and duplicate checks

_line_cols = ", ".join(f"LINE_NCH_PMT_AMT_{i}" for i in range(1, 14))
# Explicit DOUBLE cast- a line position that is empty in every row of a file is
# inferred as VARCHAR by the CSV reader and UNPIVOT refuses to mix DOUBLE and VARCHAR
_line_cast = ", ".join(f"TRY_CAST(LINE_NCH_PMT_AMT_{i} AS DOUBLE) AS LINE_NCH_PMT_AMT_{i}" for i in range(1, 14))

CARRIER_DIAG_SQL = f"""
    WITH clm_rep AS (
        SELECT COUNT(*)                 AS clm_ids_multi_row,
               COALESCE(SUM(n - 1), 0)  AS extra_rows_same_clm_id
        FROM (SELECT CLM_ID, COUNT(*) AS n FROM carrier GROUP BY CLM_ID HAVING COUNT(*) > 1)
    ),
    lines AS (
        SELECT COUNT(*) FILTER (WHERE line_pmt < 0)                     AS negative_lines,
               COALESCE(SUM(line_pmt) FILTER (WHERE line_pmt < 0), 0)   AS negative_line_dollars,
               COUNT(*) FILTER (WHERE line_pmt = 0)                     AS zero_lines
        FROM (UNPIVOT (SELECT {_line_cast} FROM carrier) ON {_line_cols} INTO NAME pos VALUE line_pmt)
    )
    SELECT * FROM clm_rep, lines
"""

PDE_DIAG_SQL = """
    WITH id_rep AS (
        SELECT COUNT(*)                 AS pde_ids_multi_row,
               COALESCE(SUM(n - 1), 0)  AS extra_rows_same_pde_id
        FROM (SELECT PDE_ID, COUNT(*) AS n FROM pde GROUP BY PDE_ID HAVING COUNT(*) > 1)
    ),
    sign AS (
        SELECT COUNT(*) FILTER (WHERE TOT_RX_CST_AMT < 0)                    AS negative_cost_rows,
               COALESCE(SUM(TOT_RX_CST_AMT) FILTER (WHERE TOT_RX_CST_AMT < 0), 0) AS negative_cost_dollars,
               COUNT(*) FILTER (WHERE TOT_RX_CST_AMT = 0)                    AS zero_cost_rows
        FROM pde
    ),
    dup AS (
        -- Identical economic content (member, date, NDC, quantity, days supply,
        -- patient pay, total cost) on more than one row, whatever the PDE_ID
        SELECT COUNT(*)                                                         AS exact_dup_groups,
               COALESCE(SUM(n - 1), 0)                                          AS exact_dup_surplus_rows,
               COALESCE(SUM(TOT_RX_CST_AMT * (n - 1)) FILTER (WHERE TOT_RX_CST_AMT > 0), 0) AS exact_dup_surplus_cost
        FROM (
            SELECT TOT_RX_CST_AMT, COUNT(*) AS n
            FROM pde
            GROUP BY DESYNPUF_ID, SRVC_DT, PROD_SRVC_ID, QTY_DSPNSD_NUM,
                     DAYS_SUPLY_NUM, PTNT_PAY_AMT, TOT_RX_CST_AMT
            HAVING COUNT(*) > 1
        )
    )
    SELECT * FROM id_rep, sign, dup
"""

with duckdb_settings(con, **HEAVY):
    diag["diag_carrier"] = store.query_df("diag_carrier", CARRIER_DIAG_SQL,
                                          inputs=f"{DESYNPUF_PATHS['carrier']}/*.csv")
    _show_df("diag_carrier", diag["diag_carrier"])
    diag["diag_pde"] = store.query_df("diag_pde", PDE_DIAG_SQL,
                                      inputs=f"{DESYNPUF_PATHS['pde']}/*.csv")
    _show_df("diag_pde", diag["diag_pde"])

print()
print(store.status().to_string(index=False))

[duckdb] SET memory_limit = '13GB'  (was '5.5 GiB')
[duckdb] SET threads = 2  (was 4)
[duckdb] SET preserve_insertion_order = False  (was True)
[artifacts] HIT   diag_carrier                             rows=             1  built=2026-11-18T08:14:45  key=e6478ca333be

  [diag_carrier]
    clm_ids_multi_row                                         0
    extra_rows_same_clm_id                                    0
    negative_lines                                            0
    negative_line_dollars                                     0
    zero_lines                                    1,091,955,632
[artifacts] HIT   diag_pde                                 rows=             1  built=2026-11-18T08:24:37  key=d39c8f682f98

  [diag_pde]
    pde_ids_multi_row                                         0
    extra_rows_same_pde_id                                    0
    negative_cost_rows                                        0
    negative_cost_dollars                                     0


In [108]:
# 1h. Segment 2 date coverage and NULL-safe duplicate recount

SEG_COVERAGE_IP_SQL = """
    SELECT SEGMENT,
           COUNT(*)                                                          AS n_rows,
           COUNT(*) FILTER (WHERE from_dt IS NULL)                           AS from_dt_null,
           COUNT(*) FILTER (WHERE thru_dt IS NULL)                           AS thru_dt_null,
           COUNT(*) FILTER (WHERE adm_dt IS NULL)                            AS adm_dt_null,
           COUNT(*) FILTER (WHERE dschrg_dt IS NULL)                         AS dschrg_dt_null,
           COUNT(*) FILTER (WHERE PRVDR_NUM IS NULL OR PRVDR_NUM = '')       AS prvdr_null,
           COUNT(*) FILTER (WHERE CLM_DRG_CD IS NULL OR CLM_DRG_CD = '')     AS drg_null,
           SUM(CLM_PMT_AMT)                                                  AS sum_pmt
    FROM clm_ip
    GROUP BY SEGMENT
"""

SEG_COVERAGE_OP_SQL = """
    SELECT SEGMENT,
           COUNT(*)                                                          AS n_rows,
           COUNT(*) FILTER (WHERE from_dt IS NULL)                           AS from_dt_null,
           COUNT(*) FILTER (WHERE thru_dt IS NULL)                           AS thru_dt_null,
           COUNT(*) FILTER (WHERE PRVDR_NUM IS NULL OR PRVDR_NUM = '')       AS prvdr_null,
           SUM(CLM_PMT_AMT)                                                  AS sum_pmt
    FROM clm_op
    GROUP BY SEGMENT
"""

# Exact-duplicate recount split by whether the grouping key had real dates.
DUP_SPLIT_SQL = """
    SELECT (from_dt IS NOT NULL AND thru_dt IS NOT NULL)                     AS dated,
           COUNT(*)                                                          AS dup_groups,
           COALESCE(SUM(n - 1), 0)                                           AS surplus_claims,
           COALESCE(SUM(CLM_PMT_AMT * (n - 1)) FILTER (WHERE CLM_PMT_AMT > 0), 0) AS surplus_dollars_positive
    FROM (
        SELECT DESYNPUF_ID, PRVDR_NUM, from_dt, thru_dt, CLM_PMT_AMT,
               COUNT(DISTINCT CLM_ID) AS n
        FROM {t}
        GROUP BY DESYNPUF_ID, PRVDR_NUM, from_dt, thru_dt, CLM_PMT_AMT
        HAVING COUNT(DISTINCT CLM_ID) > 1
    )
    GROUP BY 1
"""

with duckdb_settings(con, **HEAVY):
    for name, sql, dep in [("diag_clm_ip_segment_coverage", SEG_COVERAGE_IP_SQL, "clm_ip"),
                           ("diag_clm_op_segment_coverage", SEG_COVERAGE_OP_SQL, "clm_op")]:
        diag[name] = store.query_df(name, sql, deps=[dep], order_by="SEGMENT")
        _show_df(name, diag[name])
    for t in ["clm_ip", "clm_op"]:
        name = f"diag_{t}_dup_split"
        diag[name] = store.query_df(name, DUP_SPLIT_SQL.format(t=t), deps=[t], order_by="dated")
        _show_df(name, diag[name])

[duckdb] SET memory_limit = '13GB'  (was '5.5 GiB')
[duckdb] SET threads = 2  (was 4)
[duckdb] SET preserve_insertion_order = False  (was True)
[artifacts] HIT   diag_clm_ip_segment_coverage             rows=             2  built=2026-11-18T12:34:47  key=e452716685a3

  [diag_clm_ip_segment_coverage]
 SEGMENT  n_rows  from_dt_null  thru_dt_null  adm_dt_null  dschrg_dt_null  prvdr_null  drg_null      sum_pmt
       1 1331533             0             0            0               0           0         0 1.268515e+10
       2    1289          1289          1289            0               0           0         0 2.570824e+07
[artifacts] HIT   diag_clm_op_segment_coverage             rows=             2  built=2026-11-18T12:34:49  key=2a4fbbf67234

  [diag_clm_op_segment_coverage]
 SEGMENT   n_rows  from_dt_null  thru_dt_null  prvdr_null      sum_pmt
       1 15604184             0             0           0 4181850370.0
       2   222803        222803        222803           0  300438000.0
